# Comparative analysis

This notebook only reads completed per-video summary artifacts. It never loads classifiers or reruns video analysis. Each workflow produces a validation table before comparisons. Explicit metadata (`video_path, group, treatment, subject, region, well, day`) overrides folder-name inference.

In [ ]:
from pathlib import Path
import sys

WORKING_DIR = Path.cwd().resolve()
PROJECT_ROOT = WORKING_DIR if (WORKING_DIR / 'gcamp_analysis').is_dir() else WORKING_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
from gcamp_analysis.experiments.comparative import (
    run_hierarchical_comparison,
    run_longitudinal_comparison,
    run_treatment_comparison,
    save_comparison_tables,
)
from gcamp_analysis.reporting import save_comparisons

## Longitudinal groups

With `ALIGN=False`, this compares whole-video descriptive statistics across days and makes no cell-identity claim. `ALIGN=True` adds spatial registration, ROI matching, and cell/group tracking while retaining the descriptive tables.

In [ ]:
LONGITUDINAL_GROUPS = {
    # 'region_1': Path(r'C:\\path\\to\\region_1'),
}
LONGITUDINAL_METADATA = None  # optional CSV/XLSX/Parquet path or DataFrame
ALIGN = False
LONGITUDINAL_OUTPUT = PROJECT_ROOT / 'comparison_outputs' / 'longitudinal'

In [ ]:
longitudinal_result = None
if LONGITUDINAL_GROUPS:
    longitudinal_result = run_longitudinal_comparison(
        LONGITUDINAL_GROUPS,
        metadata=LONGITUDINAL_METADATA,
        align=ALIGN,
        output_dir=LONGITUDINAL_OUTPUT if ALIGN else None,
    )
    display(longitudinal_result.validation.to_frame())
    if not longitudinal_result.validation.has_errors:
        display(longitudinal_result.group_day_summary)
        save_comparison_tables(
            longitudinal_result.tables,
            LONGITUDINAL_OUTPUT / 'longitudinal_comparison.xlsx',
        )
else:
    print('Configure LONGITUDINAL_GROUPS to run this section.')

## Treatment groups

Set the independent experimental unit for the current experiment. Common choices are `well`, `region`, or `animal`. Longitudinal-within-treatment analysis is optional; alignment is only used when longitudinal analysis is enabled.

In [ ]:
TREATMENT_GROUPS = {
    # 'control': Path(r'C:\\path\\to\\control'),
    # 'drug': Path(r'C:\\path\\to\\drug'),
}
TREATMENT_METADATA = None
REPLICATE_UNIT = 'well'  # well, region, animal, or another metadata column
LONGITUDINAL_WITHIN_TREATMENT = False
ALIGN_WITHIN_TREATMENT = False
TREATMENT_OUTPUT = PROJECT_ROOT / 'comparison_outputs' / 'treatments'

In [ ]:
treatment_result = None
if TREATMENT_GROUPS:
    treatment_result = run_treatment_comparison(
        TREATMENT_GROUPS,
        replicate_unit=REPLICATE_UNIT,
        metadata=TREATMENT_METADATA,
        longitudinal=LONGITUDINAL_WITHIN_TREATMENT,
        align=ALIGN_WITHIN_TREATMENT,
        output_dir=TREATMENT_OUTPUT if ALIGN_WITHIN_TREATMENT else None,
    )
    display(treatment_result.validation.to_frame())
    if not treatment_result.validation.has_errors:
        display(treatment_result.treatment_summary)
        if LONGITUDINAL_WITHIN_TREATMENT:
            display(treatment_result.treatment_day_summary)
        save_comparison_tables(
            treatment_result.tables,
            TREATMENT_OUTPUT / 'treatment_comparison.xlsx',
        )
else:
    print('Configure TREATMENT_GROUPS to run this section.')

## Generic filesystem sibling comparisons

This retains the original arbitrary hierarchy behavior: immediate child folders are compared at every internal node, using persisted per-video summaries.

In [ ]:
HIERARCHICAL_ROOT = None  # e.g. Path(r'C:\\path\\to\\experiment')

if HIERARCHICAL_ROOT is not None:
    hierarchy_result = run_hierarchical_comparison(HIERARCHICAL_ROOT)
    display(hierarchy_result.validation.to_frame())
    if not hierarchy_result.validation.has_errors and hierarchy_result.root is not None:
        save_comparisons(
            root=hierarchy_result.root,
            sibling_tables=hierarchy_result.sibling_tables,
        )
        for node_path, table in hierarchy_result.sibling_tables.items():
            print(f'\nNode: {node_path}')
            display(table)
else:
    print('Set HIERARCHICAL_ROOT to run generic sibling comparisons.')